# All the same comments as for eng_pipeline

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('data/kyrgyz-constitution-2021_corpus.csv')

In [3]:
rus_sentences = data['russian_tokenized13a']

In [5]:
from datasets import load_dataset
df_kir = load_dataset('cointegrated/panlex-meanings', name='kir', split='train').to_pandas()
df_rus = load_dataset('cointegrated/panlex-meanings', name='rus', split='train').to_pandas()

Resolving data files:   0%|          | 0/7558 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/7558 [00:00<?, ?it/s]

In [6]:
def f(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return x
    return re.sub(r'\u0301', '', str(x))

In [7]:
import re
df_rus['txt'] = df_rus['txt'].apply(f)

In [8]:
df_kir_rus = df_kir.merge(df_rus, on='meaning', suffixes=['_kir','_rus'])
kir_rus_pairs = df_kir_rus[['txt_kir','txt_rus']].drop_duplicates()

In [9]:
df_rus_to_kir = df_rus.merge(df_kir, on='meaning', suffixes=['_rus','_kir'])
rus_to_kir_pairs = df_rus_to_kir[['txt_rus','txt_kir']].drop_duplicates()

In [10]:
import apertium
analyzer = apertium.Analyzer('kir')

In [11]:
kyr_sentences = data['kyrgyz_tokenized13a']
rus_sentences = data['russian_tokenized13a']

In [12]:
def parse_unit(unit):
    analyses = []
    nouns = []
    for reading in unit.readings:
        morphemes = []
        for sread in reading:
            if 'n' in sread.tags:
                morphemes.append({
                    'lemma': sread.baseform,
                    'tags': sread.tags
                })
                nouns.append(sread.baseform)
        if morphemes:
            analyses.append(morphemes)

    return {
        'surface': unit.wordform,  # some APIs use .surface_form instead
        'analyses': analyses,
        'nouns': list(set(nouns)),
    }

In [13]:
def get_nouns_from_sentence(sentence):
    units = analyzer.analyze(sentence)
    noun_lemmas = []
    for i, unit in enumerate(units):
        analysis_result = parse_unit(unit)
        analysis_result['token_id'] = i
        if analysis_result['analyses']:
            noun_lemmas.append(analysis_result)
    return noun_lemmas

In [15]:
import re
def translate(words, vocab=kir_rus_pairs,lang_source='kir', lang_target='rus'):
    matches_arrays = [vocab.loc[vocab[f"txt_{lang_source}"] == word, f"txt_{lang_target}"].unique() for word in words]
    matches = {re.sub('\u0301', '', token) for arr in matches_arrays for token in arr }
    return list(matches)

In [16]:
def add_translations(token_dct):
    for unit in token_dct:
        unit['translations'] = translate(unit['nouns'])
    return token_dct

In [44]:
def rus_align(nouns, sentence_split):
    matches = []
    for index, rus_noun, rus_parse in sentence_split:
        if rus_parse.tag.POS == 'NOUN':
            if rus_parse.normal_form in nouns:
                matches.append((index, rus_noun, rus_parse.normal_form))
    return matches

In [45]:
from ruwordnet import RuWordNet
wn = RuWordNet()

In [46]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

In [52]:
hard_cases = []
wsd_queries = []

В КОДЕ ИНДЕКС ТОКЕНА НА КЫРГЫЗСКОМ ЭТО РЕЗУЛЬТАТ ПАРСИНГА АПЕРТИУМА, ИНДЕКС НА РУССКОМ ЭТО РЕЗУЛЬТАТ СПЛИТА НА РУССКОМ

In [53]:
def kyr_wsd(sentence_id: int, src_sentences, tgt_sentences):
    sentence_src = src_sentences[sentence_id]
    sentence_tgt = tgt_sentences[sentence_id]
    nouns = get_nouns_from_sentence(sentence_src)
    nouns = add_translations(nouns)
    sentence_split = [(index, token, morph.parse(token)[0]) for index, token in enumerate(sentence_tgt.split())]
    global CNT, ALIGN, SYNSETS
    answers = []
    for unit in nouns:
        res = rus_align(unit['translations'], sentence_split)
        if res:
            lemmas = list({(ind, lemma) for ind, _, lemma in res})
            if len(lemmas) > 1:
                dct = {
                    'sentence_id': sentence_id,
                    'kyr_token_id': unit['token_id'],
                    'wordform': unit['surface'],
                    'lemmas': [lemma for _, lemma in lemmas],
                    'lemmas_ids': [ind for ind, _ in lemmas]
                }
                hard_cases.append(dct)
                continue
            #now we are sure that len(lemmas) = 1(it never equals 0, i checked it)
            dct = {
                'sentence_id': sentence_id,
                'kyr_token_id': unit['token_id'],
                'wordform': unit['surface'],
                'lemma': lemmas[0][1],
                'lemma_id': lemmas[0][0]
            }
            wsd_queries.append(dct)

In [54]:
for i in range(len(kyr_sentences)):
    kyr_wsd(i, kyr_sentences, rus_sentences)

Error: malformed input stream: Found unexpected character / unescaped in stream
: iostream error


lemma


In [91]:
final_dicts = []
for query in queries:
    sentnence_id = query['sentence_id']
    lemma = query['lemma']
    lemma_id = query['lemma_id']
    sentnence = rus_sentences[sentnence_id].split()
    sentnence.insert(lemma_id + 1, '<WSD>')
    sentnence.insert(lemma_id, '<WSD>')
    synsets = wn.get_synsets(lemma)
    synset_descriptions = ["; ".join([str(sns.id), str(sns.title), str(sns.definition)]) for sns in synsets]
    dct = {
        'sentence': " ".join(sentnence),
        'word': lemma,
        'descriptions': synset_descriptions
    }
    final_dicts.append(dct)

In [ ]:
def get_cot_prompt(word: str, sentence: str, synset_descriptions):
    parts = ['You are going to identify the corresponding sense tag of an ambiguity word in Russian sentences. Do the following tasks.', 
        f'1. {word} has different meanings. Below are possible meanings. Comprehend the sensetags and meanings.']
    parts.extend(synset_descriptions)
    parts.append('2. Now examine the sentence below. You are going to identify the most suitable meaning for ambiguity word.')
    parts.append(sentence)
    parts.extend([
        '3. Try to identify the meaning of the word in the above sentence which is enclosed with the <WSD>. You can think of the real meaning of \
sentence and decide the most suitable meaning for the word.',
        '4. Based on the identified meaning, try to find the most appropriate senseIDs from the below. You are given definition of each sense tag too.'
    ])
    parts.extend(synset_descriptions)
    parts.extend([
        '5. If you have more than one senseIDs identified after above steps, you can return the senseIDs in order of confidence level.',
        '6. Return JSON object that contains the ambiguity word and the finalized senseIDs. Use the following format for the output.',
        '<JSON Object with fields ambiguity_word and senseIDs >'
    ])
    return "\n".join(parts)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
key = os.getenv("DEEPSEEK")

client = OpenAI(
    api_key=key,
    base_url="https://api.deepseek.com",
)

In [ ]:
import tqdm

for dct in tqdm.tqdm(final_dicts):
    prompt = get_cot_prompt(dct['word'], dct['sentence'], dct['descriptions'])
    messages = [{"role": "system", "content": prompt}]
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=messages,
        response_format={
            'type': 'json_object'
        }
    )
    dct['annotation'] = json.loads(response.choices[0].message.content)

with open('deepseek_rus_results.json', 'w') as f:
    json.dump(final_dicts, f, indent=4)